# DỰ ÁN NGHIÊN CỨU IEEE: NHẬN DIỆN CĂNG THẲNG ĐÁNG TIN CẬY TỪ CẢM BIẾN CỔ TAY
## Module 2: Mô hình hóa, Hiệu chuẩn xác suất & Phân loại chọn lọc (LOSO 15-folds)

**Tác giả:** Nhóm nghiên cứu DSR301m - Đại học FPT
**Dataset:** WESAD Wrist Features (`wesad_wrist_features.csv`)
**Mục tiêu học thuật:**
1. Đánh giá kiểm thử chéo nghiêm ngặt Leave-One-Subject-Out (LOSO) trên 15 đối tượng.
2. Đo lường tỷ lệ báo động giả trong vận động (FAR_motion).
3. Hiệu chuẩn xác suất (Isotonic Calibration) và định lượng sai số ECE.
4. Thiết lập cơ chế Phân loại chọn lọc (Selective Classification) - đánh đổi Coverage lấy Selective F1.
5. Phân tích cắt bỏ (Ablation Study) chứng minh vai trò kiểm soát độ bất định của Gia tốc kế (ACC).

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
from xgboost import XGBClassifier

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Đã nạp đầy đủ các thư viện chuyên dụng!")

Đã nạp đầy đủ các thư viện chuyên dụng!


### 1. Đọc dữ liệu đặc trưng trích xuất từ WESAD

In [2]:
csv_path = "../Dataset/wesad_wrist_features.csv" if os.path.exists("../Dataset/wesad_wrist_features.csv") else "wesad_wrist_features.csv"
df = pd.read_csv(csv_path)
print(f"[*] Kích thước dữ liệu: {df.shape}")
print(f"[*] Danh sách 15 đối tượng: {df['subject_id'].unique().tolist()}")
print("\n[*] Phân phối nhãn thực nghiệm:")
print(df["label"].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))

[*] Kích thước dữ liệu: (5294, 29)
[*] Danh sách 15 đối tượng: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

[*] Phân phối nhãn thực nghiệm:
label
0    64.45%
1    35.55%
Name: proportion, dtype: object


### 2. Định nghĩa các hàm đo lường độ tin cậy (ECE & FAR_motion)

In [3]:
def compute_ece(y_true, y_prob, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        in_bin = (y_prob > bin_boundaries[i]) & (y_prob <= bin_boundaries[i+1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(y_true[in_bin])
            avg_confidence_in_bin = np.mean(y_prob[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece

def compute_far_motion(y_true, y_pred, acc_var, motion_threshold):
    motion_mask = (acc_var >= motion_threshold) & (y_true == 0)
    total_motion_negatives = np.sum(motion_mask)
    if total_motion_negatives == 0:
        return 0.0
    false_alarms = np.sum((y_pred == 1) & motion_mask)
    return (false_alarms / total_motion_negatives) * 100.0

### 3. Pipeline Đánh giá LOSO (15 Folds: Fit Scaler, Fit Model, Fit Calibrator)

In [4]:
def run_loso(df_data, include_acc=True):
    meta = ["subject_id", "time_start_sec", "time_end_sec", "label"]
    feature_cols = [c for c in df_data.columns if c not in meta]
    if not include_acc:
        feature_cols = [c for c in feature_cols if not c.startswith("acc_")]
    
    subjects = df_data["subject_id"].unique()
    records = []
    
    for test_sub in subjects:
        train_df = df_data[df_data["subject_id"] != test_sub]
        test_df = df_data[df_data["subject_id"] == test_sub]
        
        X_tr, y_tr = train_df[feature_cols].values, train_df["label"].values
        X_te, y_te = test_df[feature_cols].values, test_df["label"].values
        
        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_tr)
        X_te_sc = scaler.transform(X_te)
        
        # 1. Random Forest
        rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
        rf.fit(X_tr_sc, y_tr)
        rf_prob = rf.predict_proba(X_te_sc)[:, 1]
        
        # 2. XGBoost Raw
        xgb = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric="logloss")
        xgb.fit(X_tr_sc, y_tr)
        xgb_prob = xgb.predict_proba(X_te_sc)[:, 1]
        
        # 3. XGBoost Calibrated (Isotonic CV=3)
        cal_xgb = CalibratedClassifierCV(estimator=XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric="logloss"), method="isotonic", cv=3)
        cal_xgb.fit(X_tr_sc, y_tr)
        cal_prob = cal_xgb.predict_proba(X_te_sc)[:, 1]
        
        for i in range(len(test_df)):
            records.append({
                "subject_id": test_sub,
                "label": y_te[i],
                "acc_mag_var": test_df["acc_mag_var"].values[i],
                "rf_prob": rf_prob[i],
                "xgb_prob": xgb_prob[i],
                "cal_prob": cal_prob[i]
            })
    return pd.DataFrame(records)

motion_thresh = np.percentile(df["acc_mag_var"].values, 80)
print(f"[*] Ngưỡng phân vị thứ 80 cử động tay mạnh: {motion_thresh:.3f}")
print("[*] Đang chạy LOSO 15-folds với đầy đủ cảm biến (+ACC)...")
res_full = run_loso(df, include_acc=True)
print("[*] Đang chạy LOSO 15-folds cắt bỏ (-ACC)...")
res_no_acc = run_loso(df, include_acc=False)
print("[✓] Hoàn tất kiểm thử chéo!")

[*] Ngưỡng phân vị thứ 80 cử động tay mạnh: 15.385
[*] Đang chạy LOSO 15-folds với đầy đủ cảm biến (+ACC)...
[*] Đang chạy LOSO 15-folds cắt bỏ (-ACC)...
[✓] Hoàn tất kiểm thử chéo!


### 4. Bảng Kết quả So sánh Baseline vs Khung Phân loại Chọn lọc Đề xuất

In [5]:
def evaluate_models(res_df, motion_th):
    y_true = res_df["label"].values
    acc_v = res_df["acc_mag_var"].values
    
    rows = []
    for name, col in [("Random Forest (Gốc)", "rf_prob"), ("XGBoost (Gốc)", "xgb_prob"), ("XGBoost + Hiệu chuẩn", "cal_prob")]:
        prob = res_df[col].values
        pred = (prob >= 0.5).astype(int)
        rows.append({
            "Phương pháp": name,
            "Độ phủ (Coverage)": "100.0%",
            "Selective F1": f"{f1_score(y_true, pred, average='macro')*100:.2f}%",
            "ECE": f"{compute_ece(y_true, prob):.4f}",
            "FAR_motion": f"{compute_far_motion(y_true, pred, acc_v, motion_th):.2f}%"
        })
        
    cal_p = res_df["cal_prob"].values
    for tau in [0.70, 0.80]:
        p_tau = np.full(len(cal_p), -1)
        p_tau[cal_p >= tau] = 1
        p_tau[(1.0 - cal_p) >= tau] = 0
        acc_mask = (p_tau != -1)
        cov = np.mean(acc_mask) * 100
        f1_sel = f1_score(y_true[acc_mask], p_tau[acc_mask], average="macro") * 100
        ece_sel = compute_ece(y_true[acc_mask], cal_p[acc_mask])
        far_sel = compute_far_motion(y_true, p_tau, acc_v, motion_th)
        rows.append({
            "Phương pháp": f"Chọn lọc đề xuất (tau={tau:.2f})",
            "Độ phủ (Coverage)": f"{cov:.1f}%",
            "Selective F1": f"{f1_sel:.2f}%",
            "ECE": f"{ece_sel:.4f}",
            "FAR_motion": f"{far_sel:.2f}%"
        })
    return pd.DataFrame(rows)

df_results = evaluate_models(res_full, motion_thresh)
print(df_results.to_markdown(index=False))

| Phương pháp                 | Độ phủ (Coverage)   | Selective F1   |    ECE | FAR_motion   |
|:----------------------------|:--------------------|:---------------|-------:|:-------------|
| Random Forest (Gốc)         | 100.0%              | 89.44%         | 0.0568 | 11.19%       |
| XGBoost (Gốc)               | 100.0%              | 88.02%         | 0.0715 | 13.22%       |
| XGBoost + Hiệu chuẩn        | 100.0%              | 88.72%         | 0.0482 | 8.14%        |
| Chọn lọc đề xuất (tau=0.70) | 82.8%               | 93.70%         | 0.0299 | 2.37%        |
| Chọn lọc đề xuất (tau=0.80) | 72.6%               | 95.92%         | 0.0265 | 1.02%        |


### 5. Nghiên cứu cắt bỏ (Ablation Study) tại ngưỡng $\tau = 0.80$

In [6]:
def get_sel_row(res_df, tau=0.80):
    y_true = res_df["label"].values
    cal_p = res_df["cal_prob"].values
    p_tau = np.full(len(cal_p), -1)
    p_tau[cal_p >= tau] = 1
    p_tau[(1.0 - cal_p) >= tau] = 0
    acc_mask = (p_tau != -1)
    return {
        "Selective F1": f1_score(y_true[acc_mask], p_tau[acc_mask], average="macro") * 100,
        "ECE": compute_ece(y_true[acc_mask], cal_p[acc_mask]),
        "FAR_motion": compute_far_motion(y_true, p_tau, res_df["acc_mag_var"].values, motion_thresh)
    }

r_no = get_sel_row(res_no_acc, 0.80)
r_with = get_sel_row(res_full, 0.80)
df_ablation = pd.DataFrame([
    {"Cấu hình": "Chỉ tín hiệu sinh lý (BVP+EDA+TEMP - No ACC)", "Selective F1": f"{r_no['Selective F1']:.2f}%", "ECE": f"{r_no['ECE']:.4f}", "FAR_motion": f"{r_no['FAR_motion']:.2f}%"},
    {"Cấu hình": "Đầy đủ ngữ cảnh (+ ACC)", "Selective F1": f"{r_with['Selective F1']:.2f}%", "ECE": f"{r_with['ECE']:.4f}", "FAR_motion": f"{r_with['FAR_motion']:.2f}%"}
])
print(df_ablation.to_markdown(index=False))

| Cấu hình                                     | Selective F1   |    ECE | FAR_motion   |
|:---------------------------------------------|:---------------|-------:|:-------------|
| Chỉ tín hiệu sinh lý (BVP+EDA+TEMP - No ACC) | 93.65%         | 0.0498 | 0.68%        |
| Đầy đủ ngữ cảnh (+ ACC)                      | 95.92%         | 0.0265 | 1.02%        |
